In [2]:
from datasets import load_dataset


In [5]:
ds=load_dataset("G4KMU/t2-ragbench", "FinQA")

In [6]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'context_id', 'split', 'question', 'program_answer', 'original_answer', 'context', 'table', 'pre_text', 'post_text', 'file_name', 'company_name', 'company_symbol', 'report_year', 'page_number', 'company_sector', 'company_industry', 'company_headquarters', 'company_date_added', 'company_cik', 'company_founded'],
        num_rows: 6251
    })
    dev: Dataset({
        features: ['id', 'context_id', 'split', 'question', 'program_answer', 'original_answer', 'context', 'table', 'pre_text', 'post_text', 'file_name', 'company_name', 'company_symbol', 'report_year', 'page_number', 'company_sector', 'company_industry', 'company_headquarters', 'company_date_added', 'company_cik', 'company_founded'],
        num_rows: 883
    })
    test: Dataset({
        features: ['id', 'context_id', 'split', 'question', 'program_answer', 'original_answer', 'context', 'table', 'pre_text', 'post_text', 'file_name', 'company_name', 'company_symbol', 'r

In [8]:
print(ds)          # shows each split with its row count
print(ds.keys())   # just the split names

DatasetDict({
    train: Dataset({
        features: ['id', 'context_id', 'split', 'question', 'program_answer', 'original_answer', 'context', 'table', 'pre_text', 'post_text', 'file_name', 'company_name', 'company_symbol', 'report_year', 'page_number', 'company_sector', 'company_industry', 'company_headquarters', 'company_date_added', 'company_cik', 'company_founded'],
        num_rows: 6251
    })
    dev: Dataset({
        features: ['id', 'context_id', 'split', 'question', 'program_answer', 'original_answer', 'context', 'table', 'pre_text', 'post_text', 'file_name', 'company_name', 'company_symbol', 'report_year', 'page_number', 'company_sector', 'company_industry', 'company_headquarters', 'company_date_added', 'company_cik', 'company_founded'],
        num_rows: 883
    })
    test: Dataset({
        features: ['id', 'context_id', 'split', 'question', 'program_answer', 'original_answer', 'context', 'table', 'pre_text', 'post_text', 'file_name', 'company_name', 'company_symbol', 'r

In [9]:
import pandas as pd

In [10]:
df = pd.concat(
    [ds[name].to_pandas().assign(hf_split=name) for name in ds.keys()],
    ignore_index=True,
)

In [11]:
pd.crosstab(df["hf_split"], df["split"])

split,dev,test,train
hf_split,,,
dev,883,0,0
test,0,1147,0
train,0,0,6251


In [12]:
df["context_id"].nunique()
df["file_name"].nunique()

2789

In [13]:
df.groupby("split")[["context_id", "file_name"]].nunique()

,context_id,file_name
split,,
dev,299,299
test,380,380
train,2110,2110


In [14]:
df.groupby("context_id").size().describe()

count    2789.000000
mean        2.969165
std         1.068216
min         1.000000
25%         2.000000
50%         3.000000
75%         4.000000
max         6.000000
dtype: float64

In [15]:
#checking for data leakage (context_id)

In [16]:
train_ids = set(df.loc[df["split"] == "train", "context_id"])
dev_ids   = set(df.loc[df["split"] == "dev",   "context_id"])
test_ids  = set(df.loc[df["split"] == "test",  "context_id"])

In [17]:
print("train ∩ test:", len(train_ids & test_ids))
print("train ∩ dev: ", len(train_ids & dev_ids))
print("dev ∩ test:  ", len(dev_ids & test_ids))

train ∩ test: 0
train ∩ dev:  0
dev ∩ test:   0


In [18]:
test_df = df[df["split"] == "test"]
seen = test_df["context_id"].isin(train_ids)
print(f"{seen.sum()} of {len(test_df)} test questions ({seen.mean():.1%}) use a page seen in train")

0 of 1147 test questions (0.0%) use a page seen in train


In [19]:
#checking for data leakage (file_id)

In [22]:
train_file_ids = set(df.loc[df["split"] == "train", "file_name"])
dev_file_ids   = set(df.loc[df["split"] == "dev",   "file_name"])
test_file_ids  = set(df.loc[df["split"] == "test",  "file_name"])

In [23]:
print("train ∩ test:", len(train_file_ids & test_ids))
print("train ∩ dev: ", len(train_file_ids & dev_ids))
print("dev ∩ test:  ", len(dev_file_ids & test_ids))

train ∩ test: 0
train ∩ dev:  0
dev ∩ test:   0


In [24]:
test_df = df[df["split"] == "test"]
seen = test_df["context_id"].isin(train_ids)
print(f"{seen.sum()} of {len(test_df)} test questions ({seen.mean():.1%}) use a page seen in train")

0 of 1147 test questions (0.0%) use a page seen in train


In [25]:
df["answer_num"] = pd.to_numeric(df["program_answer"], errors="coerce")
df[df["answer_num"].isna()][["question", "program_answer"]]

,question,program_answer


In [26]:
a = df["answer_num"]
print("negative:", (a < 0).sum())
print("zero:", (a == 0).sum())
print("between -1 and 1:", a.between(-1, 1).sum())
a.describe()

negative: 1006
zero: 84
between -1 and 1: 4638


count    8.281000e+03
mean     4.390448e+09
std      3.956476e+11
min     -2.094000e+09
25%      9.995510e-02
50%      5.969973e-01
75%      2.642857e+01
max      3.600378e+13
Name: answer_num, dtype: float64

In [27]:
pct = df[df["question"].str.contains("percent|%", case=False)]
pct[["question", "program_answer"]].sample(10, random_state=0)

,question,program_answer
7418,What percentage of International Paper's capit...,0.11496531219028741
4111,What was the percentage change in net cash pro...,0.0138481291269259
5955,What percentage of Norwegian Cruise Line Holdi...,0.46055853434646077
5765,"As of September 30, 2009, what percentage of t...",0.050275267476887917
5005,What was the total interest expense for Union ...,0.5703999999999999
4039,What was the percentage decrease in American T...,-0.15730337078651685
3738,What percentage of the $88.1 million increase ...,0.6515323496027242
5222,What percentage of International Paper's consu...,0.16872586872586873
5107,"What was the percentage growth in Machinery, E...",1.1454545454545455
1495,What was the percentage increase in EOG Resour...,0.10408849915529239


In [28]:
diff = df[df["program_answer"] != df["original_answer"]]
print(len(diff))
diff[["question", "program_answer", "original_answer"]].head(10)

7927


,question,program_answer,original_answer
0,What was Analog Devices reported interest expe...,3.8,380
1,"During the fiscal year ended March 31, 2012, f...",1,NaN
2,What were the total operating expenses for Ame...,41932.20338983051,41932
3,What proportion of Intel's total cash and inve...,0.5323169340734545,53%
4,What was the percentage change in Entergy Loui...,-0.032186459489456136,-3.2%
5,What was the percentage growth in the fair val...,0.9765625,56.25%
6,What is the expected annual amortization expen...,7.385,7.4
7,"In General Mills' 2019 financial report, what ...",0.6363377271107945,63.6%
8,What percentage decrease occurred in the defer...,96.55172413793103,96.55%
9,How does the share repurchase in the quarter e...,56.57146477,56.6


In [29]:
import numpy as np

def parse_original(s):
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return np.nan
    s = str(s).strip().replace(",", "").replace("$", "")
    is_pct = s.endswith("%")
    s = s.rstrip("%").strip()
    try:
        x = float(s)
    except ValueError:
        return np.nan
    return x / 100 if is_pct else x

df["orig_num"] = df["original_answer"].apply(parse_original)
print("unparseable:", df["orig_num"].isna().sum())

unparseable: 362


In [35]:
import math

def close(a, b, tol):
    return math.isclose(a, b, rel_tol=tol, abs_tol=1e-6)

def agrees(p, o, tol=0.05):
    return close(p, o, tol) or close(p, o * 100, tol) or close(p, o / 100, tol)

In [33]:
sub = df[df["orig_num"].notna()].copy()

sub["agree"] = [
    agrees(float(p), float(o))
    for p, o in zip(sub["answer_num"], sub["orig_num"])
]

disagree = sub[~sub["agree"]]
print(f"{len(disagree)} of {len(sub)} ({len(disagree) / len(sub):.1%}) disagree")

628 of 7919 (7.9%) disagree


In [39]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

In [40]:
disagree[["question", "program_answer", "original_answer"]].sample(20, random_state=0)

,question,program_answer,original_answer
2840,"What was the percentage decline in the asset impairment loss for Republic Services from the year ended December 31, 2008, to the year ended December 31, 2009, as reflected in their consolidated statements of income?",-10.399999999999999,-70.5%
4308,What was the average annual company contribution to the American Tower Corporation's 401(k) plan from 2010 to 2012?,1.139080459770115,3.06
6773,"What was the average total stock-based compensation expense for Aon from 2008 to 2010, as reported in the company's 2010 consolidated statements of income, in millions?",329.5,218
4006,"What percentage of the $10.9 million worth of assets acquired in the GPAP Philippines acquisition, as reported in the 2014 notes to consolidated financial statements, are tangible?",11310.0,2.7%
4871,What was the highest closing sale price of DISH Network Corporation's Class A common stock on the NASDAQ Global Select Market during the first quarter of 2013?,62.42,38.02
1383,"Considering the years 2019-2020, what percentage decrease is observed in the projected benefit payments for Air Products, as outlined in the 2016 report?",-4.672897196261683,4.67%
7263,"What was the approximate price per site in thousands that American Tower paid when it purchased 800 communications sites from Vivo on March 30, 2012?",0.189625,189625
7021,"What percentage of International Paper's total future minimum commitments under existing non-cancelable operating leases and purchase obligations as of December 31, 2007, were due to lease obligations for the year 2008?",0.06510292005744375,7%
4193,"What was the change in JPMorgan Chase's net income from 2011 to 2012, as reported in the 2013 annual report, in billions?",-921.0,-1081
7285,"What is the total return if $100,000 is invested in the S&P 500 index on October 26, 2008, and sold on October 30, 2011, considering the cumulative total return as shown in the performance graph?",38289.99999999999,3829


In [41]:
docs = df.drop_duplicates("context_id")[["context_id", "context", "table"]].copy()
print(len(docs), "unique documents")

2789 unique documents


In [42]:
docs["chars"] = docs["context"].str.len()
docs["est_tokens"] = docs["chars"] / 4      # rough rule: ~4 characters per token in English
docs["est_tokens"].describe(percentiles=[0.5, 0.9, 0.95, 0.99])

count    2789.000000
mean     1130.766404
std       388.953286
min       132.750000
50%      1112.250000
90%      1578.100000
95%      1724.950000
99%      2343.370000
max      4075.750000
Name: est_tokens, dtype: float64

In [43]:
longest = docs.nlargest(3, "chars")
for _, r in longest.iterrows():
    print(r["context_id"], int(r["est_tokens"]), "tokens")
    print(r["context"][:500], "...\n")

finqa_train_ctx_978 4075 tokens
jpmorgan chase & co .
/ 2005 annual report 123 litigation reserve the firm maintains litigation reserves for certain of its litigations , including its material legal proceedings .
while the outcome of litigation is inherently uncertain , management believes , in light of all information known to it at december 31 , 2005 , that the firm 2019s litigation reserves were adequate at such date .
management reviews litigation reserves periodically , and the reserves may be increased or decreased in th ...

finqa_train_ctx_1819 3792 tokens
the company has elected the fair-value option where the interest-rate risk of such liabilities is economically hedged with derivative contracts or the proceeds are used to purchase financial assets that will also be accounted for at fair value through earnings .
the election has been made to mitigate accounting mismatches and to achieve operational simplifications .
these positions are reported in short-term borrowings and lo

In [44]:
limit = 8192  # from the docs
(docs["est_tokens"] > limit).mean()

np.float64(0.0)

In [45]:
p95 = docs["est_tokens"].quantile(0.95)
for k in [3, 5]:
    print(f"k={k}: typical ≈ {int(k * docs['est_tokens'].median())} tokens, worst-case ≈ {int(k * p95)} tokens")

k=3: typical ≈ 3336 tokens, worst-case ≈ 5174 tokens
k=5: typical ≈ 5561 tokens, worst-case ≈ 8624 tokens


In [46]:
docs = df.drop_duplicates("context_id")[
    ["context_id", "company_name", "report_year", "context", "table"]
].copy()

print(docs["company_name"].nunique(), "companies")
print(docs["report_year"].nunique(), "years")

136 companies
21 years


In [47]:
per_company = docs.groupby("company_name").size().sort_values(ascending=False)
per_company.describe()
per_company.head(10)

company_name
Entergy                      144
Union Pacific Corporation    117
JPMorgan Chase                89
American Tower                87
Goldman Sachs                 84
PNC Financial Services        81
Marathon Oil                  76
Lockheed Martin               72
Citigroup                     67
International Paper           65
dtype: int64

In [48]:
per_company_year = docs.groupby(["company_name", "report_year"]).size()
per_company_year.describe()

count    799.000000
mean       3.490613
std        3.284672
min        1.000000
25%        1.000000
50%        2.000000
75%        4.000000
max       29.000000
dtype: float64

In [53]:
test_df = df[df["split"] == "test"]
pages = test_df["company_name"].map(per_company)
print("test questions whose company has 10+ pages:", (pages >= 10).mean())

test questions whose company has 10+ pages: 0.9197907585004359


In [55]:
company = per_company.index[0]
sample = docs[docs["company_name"] == company].sort_values("report_year").head(2)
for _, r in sample.iterrows():
    print(r["report_year"], r["table"][:400], "\n")

2002 |    |                     | 2002       | 2001      |
|---:|:--------------------|:-----------|:----------|
|  0 | entergy arkansas    | $ 35.3     | $ 69.8    |
|  1 | entergy gulf states | $ 1.4      | $ 18.5    |
|  2 | entergy louisiana   | ( $ 0.3 )  | $ 8.2     |
|  3 | system energy       | ( $ 14.5 ) | ( $ 1.6 ) | 

2002 |    |   2003 | $ 1150786   |
|---:|-------:|:------------|
|  0 |   2004 | $ 925005    |
|  1 |   2005 | $ 540372    |
|  2 |   2006 | $ 139952    |
|  3 |   2007 | $ 475288    | 



In [50]:
per_company.head(10)

company_name
Entergy                      144
Union Pacific Corporation    117
JPMorgan Chase                89
American Tower                87
Goldman Sachs                 84
PNC Financial Services        81
Marathon Oil                  76
Lockheed Martin               72
Citigroup                     67
International Paper           65
dtype: int64

In [56]:
test_df = df[df["split"] == "test"]
pages = test_df["company_name"].map(per_company)
print("test questions whose company has 10+ pages:", (pages >= 10).mean())

test questions whose company has 10+ pages: 0.9197907585004359


In [57]:
for _, r in test_df.sample(8, random_state=1).iterrows():
    print("Q:", r["question"])
    print("A:", r["program_answer"])
    print(r["table"][:1000])
    print("=" * 60)

Q: What is the net change in net revenue for Entergy Mississippi, Inc. from 2015 to 2016, considering factors such as retail electric price, volume/weather, net wholesale revenue, reserve equalization, and other factors?
A: 9.100000000000023
|    |                       | amount ( in millions )   |
|---:|:----------------------|:-------------------------|
|  0 | 2015 net revenue      | $ 696.3                  |
|  1 | retail electric price | 12.9                     |
|  2 | volume/weather        | 4.7                      |
|  3 | net wholesale revenue | -2.4 ( 2.4 )             |
|  4 | reserve equalization  | -2.8 ( 2.8 )             |
|  5 | other                 | -3.3 ( 3.3 )             |
|  6 | 2016 net revenue      | $ 705.4                  |
Q: What was the net change in the gross liability for unrecognized tax benefits, excluding interest and penalties, for American Water Works in 2014, based on the provided table summarizing the changes in the company's tax positions?
A: 

## Exploration summary

**Splits**
- train 6,251 / dev 883 / test 1,147 questions (8,281 total), matching the original FinQA splits.
- The `split` column matches the Hugging Face split names (verified with a crosstab).
- No page overlap between splits: 0 of 1,147 test questions use a page seen in train.

**Answers**
- All 8,281 `program_answer` values parse as numbers.
- 1,006 negative, 84 zero, 4,638 (56%) between -1 and 1.
- Percentages are mostly written as fractions (0.115 = 11.5%), with some exceptions (e.g. 96.55).
- Decreases are usually negative, but not always.
- Labels carry floating-point noise (e.g. 9.100000000000023), so scoring uses a 1% relative tolerance.

**Label quality**
- 628 of 7,919 (7.9%) rows disagree between `program_answer` and `original_answer` (5% tolerance, allowing ×100/÷100).
- Manual review of 20 disagreements: ~7 clear `program_answer` errors (e.g. a sum instead of an average, scale errors, a question/label mismatch), plus 6 undecided.
- Estimated label noise: at least 3–5%. Likely higher: a manual read of 8 random test questions found 2 suspect labels the cross-check can't catch (both columns